# Semana 12 — QoS, tratamento de falhas e alertas no fluxo existente

Nesta aula, continuaremos usando a **Fábrica Virtual, o broker MQTT e o Node-RED que já estão executando na AWS**. Não será criada outra aplicação e nenhum componente será substituído.

Nas aulas anteriores, o fluxo foi importado, observado e depois reconstruído nó a nó. Agora ele será ampliado para responder a situações mais próximas de uma operação industrial: entrega com diferentes níveis de QoS, mensagens repetidas, dados inválidos, desconexões e alertas compreensíveis.

> O foco é o comportamento do fluxo no Node-RED. Não usaremos Python, fórmulas matemáticas, Grafana ou uma nova infraestrutura Docker.


## Objetivos de aprendizagem

Ao final da aula, você deverá ser capaz de:

- continuar o fluxo construído nas semanas anteriores, sem refazer o que já funciona;
- explicar QoS 0, 1 e 2 em termos de entrega, duplicidade e custo operacional;
- inspecionar `msg.topic`, `msg.payload`, `msg.qos` e `msg.retain`;
- separar telemetria, alarmes, produção e mensagens inesperadas;
- validar o contrato do payload antes de usar os valores;
- impedir que um mesmo evento gere duas ações;
- diferenciar erro de processamento de perda de conexão;
- gerar alertas com contexto, severidade e ação recomendada;
- apresentar o estado atual em uma página servida pelo próprio Node-RED.


![QoS, retenção, reconexão e tratamento de falhas](img/semana_12_mqtt_qos_falhas.png)

## 1. Arquitetura que já está em operação

```mermaid
flowchart LR
    U[Aluno no navegador] -->|HTTP| A[Fábrica Virtual na AWS]
    U -->|Editor web| N[Node-RED na AWS]
    A -->|publica fabrica/#| B[(Broker MQTT na AWS)]
    B -->|entrega conforme QoS| N
    N --> V[Validação e idempotência]
    V --> R[Roteamento e alertas]
    R --> F[Front servido pelo Node-RED]
```

| Componente existente | Papel nesta aula |
|---|---|
| Fábrica Virtual | gerar telemetria e cenários de superaquecimento, vibração, emergência e produção |
| Broker MQTT | receber publicações e distribuí-las aos assinantes |
| Node-RED | receber, validar, classificar, tratar duplicidades, alertar e servir a visualização |

O endereço público será informado pelo professor. Use:

- Fábrica Virtual: `http://IP_PUBLICO_DA_AWS:5000`
- Node-RED: `http://IP_PUBLICO_DA_AWS:1880`

Não altere regras de rede, contêineres ou serviços da instância compartilhada.


## 2. Verificação inicial, sem alterar o ambiente

1. Abra a Fábrica Virtual e confirme que os valores de temperatura, vibração, pressão e produção estão sendo atualizados.
2. Localize na tela o indicador de conexão com o broker.
3. Abra o Node-RED em outra aba.
4. Localize o fluxo criado na aula anterior.
5. Abra o painel **Debug**.
6. Confirme que o nó `mqtt in` está conectado e que chegam mensagens de `fabrica/#`.

### Como interpretar o resultado

- **Painel abre, mas não atualiza:** a aplicação web responde, porém o simulador pode estar pausado.
- **Simulador atualiza, mas o Debug está vazio:** confira se o fluxo foi implantado, se o `debug` está habilitado e se o `mqtt in` está conectado.
- **Nó MQTT mostra desconectado:** não recrie o broker. Registre a evidência e informe o professor, pois a infraestrutura é compartilhada.


## 3. Preservar o fluxo anterior

Antes de evoluir o fluxo:

1. selecione os nós construídos anteriormente;
2. use **Menu → Export → Selected nodes**;
3. salve o JSON como cópia de segurança;
4. duplique a aba anterior ou crie uma nova aba chamada `Semana 12 - Confiabilidade`;
5. reutilize o **configuration node** do broker que já funciona.

Não troque o host do broker por `localhost`. No ambiente AWS conteinerizado, `localhost` dentro do Node-RED aponta para o próprio contêiner do Node-RED, não para o Mosquitto. A conexão existente já representa corretamente a rede interna da aplicação.


## 4. Passo 1 — Observar a mensagem antes de tratá-la

Na aba da Semana 12:

1. arraste um nó **mqtt in**;
2. selecione a conexão MQTT já utilizada nas aulas anteriores;
3. configure o tópico como `fabrica/#`;
4. escolha inicialmente QoS `0`;
5. nomeie o nó como `Entrada MQTT - fabrica/#`;
6. conecte diretamente a um nó **debug**;
7. no `debug`, selecione **complete msg object**;
8. clique em **Deploy**.

Observe ao menos uma mensagem de cada tipo produzido pela aplicação:

- `fabrica/linha1/.../telemetria`;
- `fabrica/alarmes/...`;
- `fabrica/linha1/producao`;
- `fabrica/linha1/status`.

No painel Debug, expanda a mensagem e localize:

| Propriedade | O que informa |
|---|---|
| `msg.topic` | rota lógica usada para classificar o evento |
| `msg.payload` | conteúdo recebido; antes da conversão pode ser texto ou buffer |
| `msg.qos` | nível usado na entrega recebida pelo Node-RED |
| `msg.retain` | indica se a mensagem veio do valor retido no broker |


## 5. Passo 2 — Converter JSON de forma explícita

1. desconecte temporariamente o `debug` da entrada;
2. arraste um nó **json**;
3. configure-o para sempre converter uma string JSON em objeto JavaScript;
4. conecte `mqtt in → json → debug`;
5. clique em **Deploy**.

### Interpretação da saída

Antes do nó `json`, o payload é uma representação textual. Depois dele, as propriedades aparecem separadamente e podem ser acessadas como `msg.payload.temperatura`, `msg.payload.vibracao` e `msg.payload.maquina`.

Se o nó receber um texto que não seja JSON válido, ele gera erro. Esse comportamento será tratado com o nó `catch`; uma mensagem defeituosa não deve derrubar o restante do fluxo.


## 6. Passo 3 — Separar as rotas pelo tópico

Adicione um nó **switch** depois do `json`. Configure a propriedade avaliada como `msg.topic` e crie as saídas:

| Saída | Regra | Conteúdo esperado |
|---|---|---|
| 1 | contém `telemetria` | medições periódicas das máquinas |
| 2 | contém `alarmes` | superaquecimento, vibração e emergência |
| 3 | contém `producao` | peças boas, refugo e totais |
| 4 | contém `status` | mudanças de estado da linha |
| 5 | caso contrário | tópico ainda não mapeado |

Conecte um `debug` nomeado em cada saída. Use a Fábrica Virtual para acionar os cenários e confira se cada mensagem toma apenas o caminho esperado.

O caminho **caso contrário** é importante: ele torna visível uma mudança de contrato ou um novo tipo de evento, em vez de descartá-lo silenciosamente.


## 7. QoS: interpretação conceitual

| QoS | Compromisso de entrega | O que pode acontecer | Uso coerente nesta aplicação |
|---|---|---|---|
| 0 | melhor esforço | uma leitura pode não chegar | telemetria frequente, pois logo chegará uma nova leitura |
| 1 | pelo menos uma entrega | a mesma mensagem pode aparecer novamente | alarme importante que não pode passar despercebido |
| 2 | uma entrega no trecho MQTT | exige mais confirmações e recursos | evento raro e crítico, como emergência |

QoS não mede a qualidade do sensor e não valida o conteúdo. Uma mensagem pode chegar com QoS 2 e ainda conter um valor incorreto ou um campo ausente.

### Teste reproduzível na aplicação existente

1. Na Fábrica Virtual, localize o **Publicador Manual MQTT**.
2. Publique em `fabrica/laboratorio/qos` com QoS 0 e um payload JSON simples.
3. Repita com QoS 1 e QoS 2.
4. No Node-RED, use uma entrada temporária em `fabrica/laboratorio/#` e observe `msg.qos` no `debug` completo.
5. Compare o comportamento, mas não conclua que QoS maior significa sempre melhor. Ele custa mais comunicação e processamento.

O QoS observado pelo assinante também depende do QoS configurado na assinatura. Se a publicação usar QoS 2, mas o `mqtt in` estiver em QoS 0, a entrega ao Node-RED ocorrerá no menor nível acordado.


## 8. Passo 4 — Validar o contrato da telemetria

Na saída de telemetria, adicione um nó **function** chamado `Validar telemetria`, com duas saídas. A primeira receberá dados válidos; a segunda, rejeitados.

```javascript
const p = msg.payload || {};
const obrigatorios = ["maquina", "temperatura", "vibracao", "pressao"];
const ausentes = obrigatorios.filter(campo => p[campo] === undefined);

if (ausentes.length > 0) {
    msg.validacao = {
        valido: false,
        motivo: "campos obrigatorios ausentes",
        campos: ausentes
    };
    return [null, msg];
}

msg.validacao = { valido: true, motivo: "contrato aceito" };
return [msg, null];
```

Conecte um `debug` a cada saída. No Publicador Manual, use o tópico `fabrica/linha1/prensa_teste/telemetria` e envie:

```json
{"maquina":"PRENSA_TESTE","temperatura":72,"vibracao":3.2,"pressao":6.1}
```

Depois, envie uma versão sem `temperatura`:

```json
{"maquina":"PRENSA_TESTE","vibracao":3.2,"pressao":6.1}
```

### Saída esperada

- o primeiro payload segue para a saída válida;
- o segundo aparece na saída rejeitada com `campos: ["temperatura"]`;
- o fluxo continua processando as mensagens seguintes.


## 9. Passo 5 — Evitar processamento duplicado

QoS 1 admite retransmissão. Para testar conscientemente, use o tópico `fabrica/linha1/prensa_teste/telemetria`, selecione QoS 1 e publique duas vezes o mesmo payload no Publicador Manual:

```json
{
  "event_id":"LAB-S12-001",
  "maquina":"PRENSA_TESTE",
  "temperatura":84,
  "vibracao":5.2,
  "pressao":6.0
}
```

Antes da regra de alerta, adicione um nó **function** chamado `Controlar duplicidade`, também com duas saídas:

```javascript
const id = msg.payload?.event_id;

if (!id) {
    msg.deduplicacao = "sem event_id: nao foi possivel comparar";
    return [msg, null];
}

let processados = flow.get("eventosProcessados") || {};

if (processados[id]) {
    msg.deduplicacao = "duplicado ignorado";
    return [null, msg];
}

processados[id] = new Date().toISOString();
flow.set("eventosProcessados", processados);
msg.deduplicacao = "novo evento";
return [msg, null];
```

### Interpretação da saída

A primeira publicação deve seguir para processamento. A segunda deve aparecer na saída de duplicados. O `event_id` representa a identidade do evento; comparar somente temperatura ou horário pode confundir duas ocorrências reais com uma duplicata.


## 10. Passo 6 — Criar regras de alerta sem fórmulas

Depois da validação e do controle de duplicidade, adicione um nó **switch** chamado `Classificar condição`.

Configure as saídas em ordem:

1. `msg.payload.temperatura` maior ou igual a `80` → temperatura crítica;
2. `msg.payload.vibracao` maior ou igual a `7` → vibração crítica;
3. caso contrário → condição normal.

Em cada caminho, use nós **change** para acrescentar:

| Campo | Exemplo | Finalidade |
|---|---|---|
| `msg.alerta.severidade` | `critica` | priorização visual e operacional |
| `msg.alerta.origem` | valor de `msg.payload.maquina` | identifica o equipamento |
| `msg.alerta.mensagem` | `Temperatura acima do limite` | descreve o problema |
| `msg.alerta.acao` | `Inspecionar carga e refrigeração` | orienta a primeira resposta |

Use os botões **Superaquecimento**, **Vibração excessiva**, **E-STOP** e **Reset/Normalizar** da Fábrica Virtual. A saída deve explicar o que aconteceu; mostrar apenas a cor vermelha não é suficiente.


## 11. Passo 7 — Capturar erros com `catch`

1. arraste um nó **catch** para a aba;
2. configure-o para capturar erros dos nós `json` e `function` desta atividade;
3. conecte-o a um `debug` chamado `Erros de processamento`;
4. adicione temporariamente um nó **inject** antes do nó `json`;
5. configure o payload do `inject` como texto e use o valor `mensagem-sem-json`;
6. acione o `inject` e observe o objeto de erro.

### Como ler a saída

O `catch` deve mostrar qual nó falhou e uma descrição do erro. Ele trata falhas de processamento do fluxo. Um payload JSON válido, mas sem um campo obrigatório, deve ser tratado pela validação da seção anterior, e não pelo `catch`.


## 12. Passo 8 — Observar a conexão com `status`

1. arraste um nó **status**;
2. associe-o ao nó `Entrada MQTT - fabrica/#`;
3. ligue-o a um `debug` chamado `Estado da conexao MQTT`;
4. faça um novo **Deploy** e observe as mensagens de estado.

O nó `status` relata condições como conectado, conectando ou desconectado. Ele não substitui o `catch`:

| Mecanismo | Responde a qual pergunta? |
|---|---|
| `status` | a conexão ou o nó está operacional? |
| `catch` | algum nó gerou um erro durante o processamento? |

Como o ambiente AWS é compartilhado, **não pare o broker nem os contêineres para provocar a falha**. Se houver uma janela controlada pelo professor, apenas observe a sequência `desconectado → reconectando → conectado` e registre os horários.


## 13. Mensagem retida: estado atual, não histórico

No Publicador Manual da Fábrica Virtual:

1. use o tópico `fabrica/laboratorio/configuracao`;
2. envie `{"modo":"producao","turno":"noturno"}`;
3. selecione a opção **retain**;
4. publique;
5. somente depois, crie um novo `mqtt in` para esse tópico e conecte-o a um `debug`.

O novo assinante deve receber imediatamente a última configuração. No objeto completo, `msg.retain` ajuda a reconhecer essa origem.

### Interpretação

- retain fornece o último estado conhecido a quem acabou de se conectar;
- não guarda uma sequência de valores;
- não substitui banco de dados ou trilha de auditoria;
- deve ser usado para estado ou configuração, com cuidado para não deixar uma informação antiga parecer atual.


## 14. Last Will: o que ele realmente sinaliza

O **Last Will and Testament** é uma mensagem previamente registrada pelo cliente no broker. Se a conexão termina de forma inesperada, o broker publica essa mensagem em nome do cliente.

Interprete corretamente:

- `offline` via Last Will indica perda inesperada da sessão MQTT;
- isso não prova, sozinho, que a máquina física quebrou;
- a causa pode ser rede, processo, sistema operacional, reinício ou falha do equipamento;
- o alerta deve informar **perda de comunicação** e solicitar verificação, sem inventar uma causa.

Nesta aula, o conceito será observado no fluxo existente se o cliente da aplicação já possuir Last Will configurado. Não interrompa a aplicação AWS compartilhada para forçar o evento.


## 15. Passo 9 — Guardar o estado para o front do Node-RED

Na saída válida da telemetria, acrescente um nó **change**:

- regra: **set** `flow.ultimaTelemetria`;
- valor: `msg.payload`;
- tipo: `msg`.

Na saída de alerta, acrescente outro **change**:

- regra: **set** `flow.ultimoAlerta`;
- valor: `msg.alerta`;
- tipo: `msg`.

O contexto `flow` funciona como memória compartilhada pelos nós da mesma aba. Ele permite que uma requisição HTTP leia o último estado mesmo quando nenhuma mensagem MQTT chega exatamente naquele instante.

O contexto padrão costuma ficar em memória. Portanto, após reiniciar o Node-RED, os valores podem desaparecer. Isso é aceitável para esta visualização didática, mas deve ser considerado em um sistema real.


## 16. Passo 10 — Criar uma página no próprio Node-RED

Esta página usa apenas os nós principais do Node-RED, sem Grafana.

### 16.1 Entrada HTTP

1. adicione um nó **http in**;
2. selecione o método `GET`;
3. informe a rota `/semana12/monitor`;
4. conecte-o a um nó **function** chamado `Montar modelo da tela`.

```javascript
msg.payload = {
    telemetria: flow.get("ultimaTelemetria") || null,
    alerta: flow.get("ultimoAlerta") || null,
    atualizadoEm: new Date().toLocaleString("pt-BR")
};
return msg;
```

### 16.2 Template HTML

Depois da função, adicione um nó **template** com o conteúdo:

```html
<!doctype html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <meta http-equiv="refresh" content="5">
  <title>Monitor Industrial</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 2rem; background: #f3f4f6; }
    .card { background: white; padding: 1rem; margin-bottom: 1rem; border-radius: 8px; }
    .alerta { border-left: 6px solid #b91c1c; }
    dt { font-weight: bold; } dd { margin: 0 0 .6rem 0; }
  </style>
</head>
<body>
  <h1>Monitor da linha</h1>
  <p>Atualizado em {{payload.atualizadoEm}}</p>

  <section class="card">
    <h2>Última telemetria</h2>
    {{#payload.telemetria}}
    <dl>
      <dt>Máquina</dt><dd>{{maquina}}</dd>
      <dt>Temperatura</dt><dd>{{temperatura}} °C</dd>
      <dt>Vibração</dt><dd>{{vibracao}} mm/s</dd>
      <dt>Pressão</dt><dd>{{pressao}} bar</dd>
    </dl>
    {{/payload.telemetria}}
    {{^payload.telemetria}}<p>Aguardando telemetria...</p>{{/payload.telemetria}}
  </section>

  <section class="card alerta">
    <h2>Último alerta</h2>
    {{#payload.alerta}}
      <p><strong>{{severidade}}</strong>: {{mensagem}}</p>
      <p>Origem: {{origem}}</p>
      <p>Ação: {{acao}}</p>
    {{/payload.alerta}}
    {{^payload.alerta}}<p>Nenhum alerta registrado.</p>{{/payload.alerta}}
  </section>
</body>
</html>
```

### 16.3 Resposta HTTP

1. conecte o `template` a um nó **http response**;
2. clique em **Deploy**;
3. abra `http://IP_PUBLICO_DA_AWS:1880/semana12/monitor`.

O front deve mostrar valores com unidades, origem do alerta e ação recomendada. A atualização automática ocorre a cada cinco segundos.


## 17. Sequência de validação

Execute os testes na ordem e registre a saída observada.

| Teste | Ação na aplicação AWS | Resultado esperado no Node-RED |
|---|---|---|
| T1 | aguardar telemetria normal | rota de telemetria, contrato válido e monitor atualizado |
| T2 | acionar superaquecimento | alerta crítico com máquina, motivo e ação |
| T3 | acionar vibração excessiva | alerta de vibração no caminho correto |
| T4 | acionar E-STOP | evento de emergência separado da telemetria |
| T5 | publicar JSON sem temperatura | mensagem rejeitada com o campo ausente identificado |
| T6 | publicar duas vezes `LAB-S12-001` | primeiro evento processado e segundo marcado como duplicado |
| T7 | injetar texto inválido antes do nó `json` | erro visível na saída do `catch` |
| T8 | publicar configuração retida | novo assinante recebe imediatamente o último estado |
| T9 | resetar/normalizar a planta | telemetria retorna ao caminho normal |

Para cada teste, responda:

1. Qual foi o tópico?
2. Qual caminho a mensagem percorreu?
3. Que saída comprovou o comportamento?
4. O operador entenderia o problema e a ação esperada?


## 18. Exercícios práticos

### Exercício 1 — Decisão de QoS

Escolha e justifique QoS 0, 1 ou 2 para:

- temperatura publicada continuamente;
- alarme de superaquecimento;
- contagem de produção;
- parada de emergência;
- configuração do modo de operação.

Explique em palavras o efeito de perda, duplicidade e maior troca de confirmações.

### Exercício 2 — Mensagem de alerta útil

Melhore os nós `change` para que todo alerta apresente severidade, origem, valor observado, unidade, instante e ação recomendada.

### Exercício 3 — Contadores do fluxo

Use o contexto do fluxo para contar mensagens válidas, inválidas e duplicadas. Acrescente esses totais ao monitor HTTP.

### Exercício 4 — Recuperação operacional

Crie um estado `normalizado` quando a planta retornar à condição normal depois de um alerta. A tela deve diferenciar “nunca houve alerta” de “houve alerta e a condição foi normalizada”.

### Exercício 5 — Análise de falha

Para JSON inválido, campo ausente, mensagem duplicada e MQTT desconectado, descreva:

- como o fluxo detecta a condição;
- qual saída deve aparecer;
- o que não deve ser processado;
- qual seria a primeira ação de diagnóstico.


## 19. Entrega da aula

Entregue:

1. JSON exportado da aba `Semana 12 - Confiabilidade`;
2. captura do fluxo completo, com os nós nomeados;
3. captura do front em condição normal;
4. captura do front com um alerta;
5. evidências dos testes T5, T6, T7 e T8;
6. um texto curto explicando por que QoS não elimina a necessidade de validação e idempotência.

Antes de encerrar, exporte novamente o fluxo. Não pare nem remova serviços da AWS compartilhada.


## Bibliografia e documentação

- NODE-RED. **Node-RED Concepts**. <https://nodered.org/docs/user-guide/concepts>.
- NODE-RED. **Working with messages**. <https://nodered.org/docs/user-guide/messages>.
- NODE-RED. **The Core Nodes**. <https://nodered.org/docs/user-guide/nodes>.
- NODE-RED. **Writing Functions**. <https://nodered.org/docs/user-guide/writing-functions>.
- NODE-RED. **Using context in a Function node**. <https://nodered.org/docs/user-guide/writing-functions#storing-data>.
- OASIS. **MQTT Version 5.0**. <https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html>.
- MQTT.org. **MQTT: The Standard for IoT Messaging**. <https://mqtt.org/>.
- ECLIPSE MOSQUITTO. **Documentation**. <https://mosquitto.org/documentation/>.

### Síntese

QoS cuida da entrega MQTT, mas não garante que o dado está correto nem que a aplicação o processará uma única vez. O fluxo confiável combina inspeção, conversão, validação, idempotência, tratamento de erros, observação da conexão e uma saída que faça sentido para quem opera o processo.
